# Heterogeneous message-direction sweep (`gs_hmd`)

This notebook analyses the `heterogenous_message_direction` folder: the full
3-by-3 factorial of generator and load attachment directions in the direct
heterogeneous graph, with three seeds each (27 runs).

Every run is otherwise identical -- `heterogeneous` graph, no input
preprocessing, GINE, mean pooling, no substation edges or nodes, and a
15M-step budget -- so `generator_direction` and `load_direction` are the only
screened factors.

The baseline is `bidirectional / bidirectional` (`gbi_lbi`), and the main
views are:

- aggregated episodic-survival curves for all nine direction pairs;
- the same curves expressed as a **difference from the baseline**;
- absolute and baseline-relative heatmaps over the direction grid, using
  the deterministic full-test evaluation of each run's saved best checkpoint.

In [1]:
from pathlib import Path
import importlib
import json
import sys
import tomllib

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

for candidate in [Path.cwd(), *Path.cwd().parents]:
    helpers = candidate / "helpers"
    repo_helpers = candidate / "Topology_Task" / "analysis" / "metrics" / "helpers"
    if helpers.exists() and (helpers / "wandb_metrics.py").exists():
        sys.path.insert(0, str(helpers))
        break
    if repo_helpers.exists() and (repo_helpers / "wandb_metrics.py").exists():
        sys.path.insert(0, str(repo_helpers))
        break
else:
    raise FileNotFoundError(
        "Could not locate Topology_Task/analysis/metrics/helpers"
    )

import wandb_metrics as wm
wm = importlib.reload(wm)
import survival_comparison as sc
sc = importlib.reload(sc)
print("wandb_metrics:", wm.__file__)
print("survival_comparison:", sc.__file__)
print("task directory:", wm.TASK_DIR)
print("done")

wandb_metrics: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/analysis/metrics/helpers/wandb_metrics.py
survival_comparison: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/analysis/metrics/helpers/survival_comparison.py
task directory: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task
done


## Analysis controls

`COMPARISON_BUDGET_STEPS = None` compares every run at the largest step that
**all** selected runs reached. That matters here: these runs are configured for
15M steps but stop early when `time_limit` (2880 minutes) expires, so the
achieved endpoint varies per run and per cluster. Set an explicit value to pin
the comparison point instead.

Endpoint tables, heatmaps, and rankings use the downloaded deterministic
full-test result of each run's `best_test_` checkpoint. Each result covers all
201 test chronics; there is no averaging over the final training evaluations.

In [2]:
USE_LOCAL_CACHE_ONLY = True
SMOOTH_WINDOW = 5
TARGET_BUDGET_STEPS = 15_000_000
COMPARISON_BUDGET_STEPS = None
FULL_TEST_EVAL_DIR = wm.TASK_DIR / "outputs" / "full_test_eval"
CONTINUATION_FULL_TEST_EVAL_DIR = FULL_TEST_EVAL_DIR / "continuation_20260731"
EXPECTED_FULL_TEST_EPISODES = 201
PREFER_CONTINUATION_FULL_TEST = True

HMD_DIR = (
    wm.TASK_DIR / "configs" / "gnn_graph_screening"
    / "heterogenous_message_direction"
)

# Short codes matching the config file names.
DIRECTION_CODES = {
    "bidirectional": "bi",
    "asset_to_busbar": "a2b",
    "busbar_to_asset": "b2a",
}
DIRECTION_ORDER = ["bidirectional", "asset_to_busbar", "busbar_to_asset"]

BASELINE_GENERATOR_DIRECTION = "bidirectional"
BASELINE_LOAD_DIRECTION = "bidirectional"
BASELINE_LABEL = "g=bi · l=bi"

FACTOR_COLUMNS = ["generator_direction", "load_direction"]


# Curve label for plot_survival_comparison. This may only read columns passed
# to group_by: labels are built from summary[group_columns], so passing the
# catalog's own "direction_label" column as label_by would raise KeyError.
def direction_curve_label(row):
    return f'g={row["generator_code"]} · l={row["load_code"]}'


print("config folder:", HMD_DIR)
print("full-test results:", FULL_TEST_EVAL_DIR)
print("continuation full-test results:", CONTINUATION_FULL_TEST_EVAL_DIR)
print("prefer continuation:", PREFER_CONTINUATION_FULL_TEST)
print("baseline:", BASELINE_LABEL)
print("done")

config folder: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/configs/gnn_graph_screening/heterogenous_message_direction
full-test results: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/full_test_eval
continuation full-test results: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/full_test_eval/continuation_20260731
prefer continuation: True
baseline: g=bi · l=bi
done


## Build the run catalog from TOML

Settings are read from the config files rather than parsed out of file names,
so the catalog stays correct if a file is renamed or reseeded.

In [3]:
def direction_code(value):
    return DIRECTION_CODES.get(str(value), str(value))


def config_record(path):
    with path.open("rb") as file:
        config = tomllib.load(file)
    args = config["args"]
    run = config.get("run", {})
    record = {
        "config_path": str(path.relative_to(wm.TASK_DIR)),
        "config": path.name,
        "run_name": str(run.get("name", path.stem)),
        "seed": int(args.get("seed", 0)),
        "declared_cuda": bool(args.get("cuda", False)),
        "graph_type": str(args.get("gnn_graph_type", "bus")),
        "encoder": str(args.get("gnn_type", "gine")),
        "generator_direction": str(
            args.get("gnn_generator_edge_direction", "bidirectional")
        ),
        "load_direction": str(
            args.get("gnn_load_edge_direction", "bidirectional")
        ),
        "line_direction": str(
            args.get("gnn_line_node_edge_direction", "bidirectional")
        ),
        "summary_direction": str(
            args.get("gnn_summary_edge_direction", "bidirectional")
        ),
        "configured_steps": int(args.get("total_timesteps", TARGET_BUDGET_STEPS)),
        "time_limit_minutes": float(args.get("time_limit", np.nan)),
    }
    record["generator_code"] = direction_code(record["generator_direction"])
    record["load_code"] = direction_code(record["load_direction"])
    record["direction_label"] = (
        f'g={record["generator_code"]} · l={record["load_code"]}'
    )
    record["factor_key"] = "|".join(
        str(record[column]) for column in FACTOR_COLUMNS
    )
    record["is_baseline"] = (
        record["generator_direction"] == BASELINE_GENERATOR_DIRECTION
        and record["load_direction"] == BASELINE_LOAD_DIRECTION
    )
    return record


if not HMD_DIR.exists():
    raise FileNotFoundError(f"Missing config folder: {HMD_DIR}")

config_catalog = pd.DataFrame(
    [config_record(path) for path in sorted(HMD_DIR.glob("*.toml"))]
)
if config_catalog.empty:
    raise RuntimeError(f"No TOML files were found in {HMD_DIR}.")

print(f"Cataloged {len(config_catalog)} declared runs.")

# Anything unexpectedly non-constant would confound the direction screen.
held_constant = [
    "graph_type",
    "encoder",
    "line_direction",
    "summary_direction",
    "configured_steps",
]
for column in held_constant:
    values = sorted(config_catalog[column].astype(str).unique())
    flag = "" if len(values) == 1 else "  <-- NOT CONSTANT"
    print(f"  {column}: {values}{flag}")

display(
    config_catalog.pivot_table(
        index="generator_code",
        columns="load_code",
        values="seed",
        aggfunc="count",
        fill_value=0,
    ).rename_axis(index="generator", columns="load")
)
print("done")

Cataloged 27 declared runs.
  graph_type: ['heterogeneous']
  encoder: ['gine']
  line_direction: ['bidirectional']
  summary_direction: ['bidirectional']
  configured_steps: ['15000000']


load,a2b,b2a,bi
generator,,,
a2b,3,3,3
b2a,3,3,3
bi,3,3,3


done


## Load the W&B histories

Only the 27 run names declared by the folder are selected. The compute backend
is taken from the effective downloaded run configuration (`cuda=True` means the
IZAR/GPU cluster), because the launch script can override the TOML.

In [4]:
requested_run_names = config_catalog["run_name"].drop_duplicates().tolist()
requested_run_name_set = set(requested_run_names)
wm.configure_run_filter_from_names(requested_run_names)
data = wm.load_wandb_data(use_local_cache_only=USE_LOCAL_CACHE_ONLY)

runs_df = data.runs_df.copy()
history_df = data.history_df.copy()
if not history_df.empty:
    history_df = history_df[
        history_df["run_name"].astype(str).isin(requested_run_name_set)
    ].copy()
if "name" in runs_df:
    runs_df = runs_df[
        runs_df["name"].astype(str).isin(requested_run_name_set)
    ].copy()


def parse_optional_bool(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, str):
        normalized = value.strip().lower()
        if normalized in {"true", "1", "yes", "y"}:
            return True
        if normalized in {"false", "0", "no", "n"}:
            return False
    return bool(value)


if "cuda" in runs_df and "name" in runs_df:
    runtime_backend = runs_df[["name", "cuda"]].copy()
    runtime_backend["runtime_cuda"] = runtime_backend["cuda"].map(
        parse_optional_bool
    )
    runtime_backend = (
        runtime_backend.dropna(subset=["runtime_cuda"])
        .drop_duplicates("name", keep="last")
        .rename(columns={"name": "run_name"})[["run_name", "runtime_cuda"]]
    )
    config_catalog = config_catalog.merge(
        runtime_backend, on="run_name", how="left"
    )
else:
    config_catalog["runtime_cuda"] = np.nan
config_catalog["runtime_cuda"] = config_catalog["runtime_cuda"].where(
    config_catalog["runtime_cuda"].notna(), config_catalog["declared_cuda"]
)
config_catalog["runtime_cuda"] = config_catalog["runtime_cuda"].astype(bool)
config_catalog["compute_backend"] = np.where(
    config_catalog["runtime_cuda"],
    "IZAR (GPU)",
    "JED (CPU)",
)

found_run_names = set(history_df.get("run_name", pd.Series(dtype=str)))
print(
    f"Loaded histories for {len(found_run_names)} / "
    f"{len(requested_run_names)} declared runs."
)
missing_run_names = sorted(requested_run_name_set - found_run_names)
if missing_run_names:
    print("Missing histories:")
    for name in missing_run_names:
        print("  ", name)
print("done")

Explicit run-name filter: 27 candidates
Project: corentin-plumet-epfl/Grid2Op
Task dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task
Cache mode: full
Cache dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache
Local-only mode: True
Force refresh: False
Refresh scan-history fallbacks: False
Selected 27 cached runs from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache/full_history
state
finished    27
History artifact setup: local_only=True, runs_df=27
[ 1/27] loading artifact cache: gs_hmd_hetero_n0_none_ga2b_la2b_s0
    loaded 361 rows, 113 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/gs_hmd/runs/gs_hmd_hetero_n0_none_ga2b_la2b_s0__MAPPO_bus14_T_0_0__I__1784891382_22267/history.parquet in 0.1s
[ 2/27] loading artifact cache: gs_hmd_hetero_n0_none_ga2b_la2b_s1
    loaded 361 rows, 113 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_dat

/Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/analysis/metrics/helpers/wandb_metrics.py:626: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out = pd.concat(pieces, ignore_index=True, sort=False)


[ 9/27] loading artifact cache: gs_hmd_hetero_n0_none_ga2b_lbi_s2
    loaded 361 rows, 113 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/gs_hmd/runs/gs_hmd_hetero_n0_none_ga2b_lbi_s2__MAPPO_bus14_T_2_0__I__1784891382_32503/history.parquet in 0.0s
[10/27] loading artifact cache: gs_hmd_hetero_n0_none_gb2a_la2b_s0
    loaded 361 rows, 113 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/gs_hmd/runs/gs_hmd_hetero_n0_none_gb2a_la2b_s0__MAPPO_bus14_T_0_0__I__1784906743_15769/history.parquet in 0.0s
[11/27] loading artifact cache: gs_hmd_hetero_n0_none_gb2a_la2b_s1
    loaded 361 rows, 113 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/gs_hmd/runs/gs_hmd_hetero_n0_none_gb2a_la2b_s1__MAPPO_bus14_T_1_0__I__1784906757_10978/history.parquet in 0.0s
[12/27] loading artifact cache: gs_hmd_hetero_n0_none_gb2a_la2b_s2
    loaded 361 rows, 113 columns from /Users/corentinplumet

## Coverage

Because `time_limit` can stop a run before the 15M-step budget, check how far
each run actually got before trusting any endpoint comparison. A wide spread
here -- especially one that lines up with the compute backend -- is itself a
finding.

In [5]:
if history_df.empty:
    raise RuntimeError(
        "No history was loaded. Download the gs_hmd runs first, or set "
        "USE_LOCAL_CACHE_ONLY = False."
    )

observed_progress = (
    history_df.groupby("run_name", as_index=False)["step"]
    .max()
    .rename(columns={"step": "observed_steps"})
)
coverage = config_catalog.merge(observed_progress, on="run_name", how="left")
coverage["observed_steps_m"] = coverage["observed_steps"] / 1_000_000
coverage["completion_pct"] = (
    100 * coverage["observed_steps"] / coverage["configured_steps"]
)
coverage["has_history"] = coverage["observed_steps"].notna()

analysis_catalog = coverage[coverage["has_history"]].copy()

with pd.option_context("display.max_colwidth", None):
    display(
        coverage.sort_values(["generator_code", "load_code", "seed"])[
            [
                "direction_label",
                "seed",
                "run_name",
                "compute_backend",
                "observed_steps_m",
                "completion_pct",
                "has_history",
            ]
        ].round(2)
    )

print("Observed-step spread by backend:")
display(
    analysis_catalog.groupby("compute_backend", as_index=False).agg(
        runs=("run_name", "nunique"),
        min_steps_m=("observed_steps_m", "min"),
        median_steps_m=("observed_steps_m", "median"),
        max_steps_m=("observed_steps_m", "max"),
    ).round(2)
)

progress_plot = px.bar(
    coverage.sort_values("observed_steps_m"),
    x="observed_steps_m",
    y="run_name",
    color="compute_backend",
    orientation="h",
    hover_data=["direction_label", "seed"],
    title="gs_hmd run coverage",
    labels={
        "observed_steps_m": "Observed environment steps (millions)",
        "run_name": "Run",
    },
    height=max(600, 20 * len(coverage)),
)
progress_plot.add_vline(
    x=TARGET_BUDGET_STEPS / 1_000_000,
    line_dash="dash",
    annotation_text="15M target",
)
progress_plot.show()
print("done")

,direction_label,seed,run_name,compute_backend,observed_steps_m,completion_pct,has_history
0,g=a2b · l=a2b,0,gs_hmd_hetero_n0_none_ga2b_la2b_s0,IZAR (GPU),14.97,99.81,True
1,g=a2b · l=a2b,1,gs_hmd_hetero_n0_none_ga2b_la2b_s1,JED (CPU),14.97,99.81,True
2,g=a2b · l=a2b,2,gs_hmd_hetero_n0_none_ga2b_la2b_s2,IZAR (GPU),14.97,99.81,True
3,g=a2b · l=b2a,0,gs_hmd_hetero_n0_none_ga2b_lb2a_s0,IZAR (GPU),14.97,99.81,True
4,g=a2b · l=b2a,1,gs_hmd_hetero_n0_none_ga2b_lb2a_s1,JED (CPU),14.97,99.81,True
5,g=a2b · l=b2a,2,gs_hmd_hetero_n0_none_ga2b_lb2a_s2,JED (CPU),14.72,98.15,True
6,g=a2b · l=bi,0,gs_hmd_hetero_n0_none_ga2b_lbi_s0,JED (CPU),14.97,99.81,True
7,g=a2b · l=bi,1,gs_hmd_hetero_n0_none_ga2b_lbi_s1,JED (CPU),14.97,99.81,True
8,g=a2b · l=bi,2,gs_hmd_hetero_n0_none_ga2b_lbi_s2,IZAR (GPU),14.97,99.81,True
9,g=b2a · l=a2b,0,gs_hmd_hetero_n0_none_gb2a_la2b_s0,JED (CPU),14.97,99.81,True


Observed-step spread by backend:


,compute_backend,runs,min_steps_m,median_steps_m,max_steps_m
0,IZAR (GPU),12,14.52,14.97,14.97
1,JED (CPU),15,14.43,14.97,14.97


done


## Extract test episodic-survival curves

`extract_survival_curves` picks the first available test-survival metric per
run and converts it to percentage points. Smoothing is left at 1 here so each
plot below can choose its own window.

In [6]:
survival_long = sc.extract_survival_curves(
    history_df,
    catalog=analysis_catalog,
    split="test",
    smooth=1,
)
if survival_long.empty:
    raise RuntimeError("No test episodic-survival history was found.")

print("metrics used:", sorted(survival_long["metric"].unique()))
print(f"{survival_long['run_name'].nunique()} runs with survival curves")

max_survival_steps = survival_long.groupby("run_name")["step"].max()
automatic_common_budget = int(max_survival_steps.min())
comparison_budget = int(
    COMPARISON_BUDGET_STEPS
    if COMPARISON_BUDGET_STEPS is not None
    else automatic_common_budget
)
print(
    "Comparison budget:",
    f"{comparison_budget / 1_000_000:.3f}M steps",
    "(automatic common budget)"
    if COMPARISON_BUDGET_STEPS is None
    else "(user selected)",
)
if COMPARISON_BUDGET_STEPS is None:
    slowest = max_survival_steps.idxmin()
    print(f"  set by the shortest run: {slowest}")
print("done")

metrics used: ['test/charts/episodic_survival']
27 runs with survival curves
Comparison budget: 14.432M steps (automatic common budget)
  set by the shortest run: gs_hmd_hetero_n0_none_gbi_la2b_s1
done


## Full-test result of each run's best checkpoint

Each JSON file was produced by evaluating one saved `best_test_` checkpoint
deterministically over all 201 held-out test chronics. The checkpoint name is
used to match the result to its declared run. Missing or duplicate results are
treated as errors rather than silently falling back to a training-history
endpoint.

In [7]:
def best_checkpoint_run_name(record, result_path=None):
    checkpoint_stem = Path(str(record.get("checkpoint", ""))).stem
    if not checkpoint_stem and result_path is not None:
        checkpoint_stem = result_path.stem
    prefix = "best_test_"
    if not checkpoint_stem.startswith(prefix):
        raise ValueError(
            f"Expected a best_test_ checkpoint, got {checkpoint_stem!r}."
        )
    checkpoint_stem = checkpoint_stem[len(prefix):]
    # Fallback for default output-json filenames when the checkpoint field is missing.
    return checkpoint_stem.split("_step", 1)[0]


def load_full_test_results(eval_dir, source_label, source_priority, recursive=False):
    pattern = "best_test_gs_hmd_*.json"
    paths = sorted(
        eval_dir.rglob(pattern)
        if recursive and eval_dir.exists()
        else eval_dir.glob(pattern)
        if eval_dir.exists()
        else []
    )
    print(f"Found {len(paths)} {source_label} full-test JSON file(s) under {eval_dir}")
    rows = []
    for result_path in paths:
        with result_path.open("r", encoding="utf-8") as file:
            record = json.load(file)
        run_name = best_checkpoint_run_name(record, result_path=result_path)
        if run_name not in requested_run_name_set:
            continue
        rows.append(
            {
                "run_name": run_name,
                "best_eval_source": source_label,
                "best_eval_source_priority": source_priority,
                "best_eval_step": int(record["checkpoint_global_step"]),
                "best_eval_survival_pct": float(record["survival_percent"]),
                "best_eval_episodes": int(record["eval_episodes"]),
                "best_eval_split": str(record["split"]),
                "best_eval_deterministic": bool(record["deterministic_eval"]),
                "best_eval_created_at": record.get("created_at", ""),
                "best_eval_json": str(result_path.relative_to(wm.TASK_DIR)),
            }
        )
    frame = pd.DataFrame(rows)
    if frame.empty:
        return frame
    frame["best_eval_created_at_ts"] = pd.to_datetime(
        frame["best_eval_created_at"], errors="coerce", utc=True
    )
    duplicate_source_runs = frame[frame.duplicated("run_name", keep=False)]
    if not duplicate_source_runs.empty:
        print(
            f"Warning: duplicate {source_label} full-test results found; "
            "keeping the latest created_at / highest-step file per run."
        )
        frame = (
            frame.sort_values(
                ["run_name", "best_eval_created_at_ts", "best_eval_step", "best_eval_json"]
            )
            .drop_duplicates("run_name", keep="last")
            .reset_index(drop=True)
        )
    return frame.drop(columns=["best_eval_created_at_ts"])


eval_sources = [
    ("old", FULL_TEST_EVAL_DIR, 0, False),
    (
        "continuation",
        CONTINUATION_FULL_TEST_EVAL_DIR,
        1 if PREFER_CONTINUATION_FULL_TEST else -1,
        False,
    ),
]
full_test_frames = [
    load_full_test_results(eval_dir, source_label, priority, recursive=recursive)
    for source_label, eval_dir, priority, recursive in eval_sources
]
full_test_all_results = pd.concat(
    [frame for frame in full_test_frames if not frame.empty],
    ignore_index=True,
    sort=False,
)
if full_test_all_results.empty:
    raise RuntimeError(
        f"No gs_hmd full-test JSON files found under {FULL_TEST_EVAL_DIR} "
        "or the continuation folder."
    )

full_test_results = (
    full_test_all_results.sort_values(
        [
            "run_name",
            "best_eval_source_priority",
            "best_eval_created_at",
            "best_eval_step",
            "best_eval_json",
        ]
    )
    .drop_duplicates("run_name", keep="last")
    .drop(columns=["best_eval_source_priority"])
    .reset_index(drop=True)
)

selection_summary = (
    full_test_results.groupby("best_eval_source", as_index=False)
    .agg(runs=("run_name", "nunique"))
    .sort_values("best_eval_source")
)
print("Selected full-test source counts:")
display(selection_summary)

old_vs_continuation = (
    full_test_all_results.pivot_table(
        index="run_name",
        columns="best_eval_source",
        values=["best_eval_survival_pct", "best_eval_step"],
        aggfunc="first",
    )
)
old_vs_continuation.columns = [f"{metric}_{source}" for metric, source in old_vs_continuation.columns]
old_vs_continuation = old_vs_continuation.reset_index().merge(
    config_catalog[["run_name", "direction_label", "seed"]],
    on="run_name",
    how="left",
)
if {
    "best_eval_survival_pct_old",
    "best_eval_survival_pct_continuation",
}.issubset(old_vs_continuation.columns):
    old_vs_continuation["continuation_delta_pp"] = (
        old_vs_continuation["best_eval_survival_pct_continuation"]
        - old_vs_continuation["best_eval_survival_pct_old"]
    )
    old_vs_continuation["continuation_step_delta_m"] = (
        old_vs_continuation["best_eval_step_continuation"]
        - old_vs_continuation["best_eval_step_old"]
    ) / 1_000_000
    with pd.option_context("display.max_colwidth", None):
        display(
            old_vs_continuation.dropna(
                subset=[
                    "best_eval_survival_pct_old",
                    "best_eval_survival_pct_continuation",
                ]
            ).sort_values(["direction_label", "seed"])[
                [
                    "direction_label",
                    "seed",
                    "run_name",
                    "best_eval_step_old",
                    "best_eval_survival_pct_old",
                    "best_eval_step_continuation",
                    "best_eval_survival_pct_continuation",
                    "continuation_delta_pp",
                    "continuation_step_delta_m",
                ]
            ].round(2)
        )

found_full_test_runs = set(full_test_results["run_name"])
missing_full_test_runs = sorted(requested_run_name_set - found_full_test_runs)
if missing_full_test_runs:
    raise ValueError(
        "Missing full-test best-checkpoint results for: "
        + ", ".join(missing_full_test_runs)
    )
if not full_test_results["best_eval_split"].eq("test").all():
    raise ValueError("At least one full-test result does not use split='test'.")
if not full_test_results["best_eval_deterministic"].all():
    raise ValueError("At least one full-test result is not deterministic.")
if not full_test_results["best_eval_episodes"].eq(
    EXPECTED_FULL_TEST_EPISODES
).all():
    raise ValueError(
        "Full-test episode counts differ from the expected complete test split: "
        f"{sorted(full_test_results['best_eval_episodes'].unique())}"
    )

endpoint_df = coverage.merge(full_test_results, on="run_name", how="inner")
endpoint_df["best_eval_step_m"] = endpoint_df["best_eval_step"] / 1_000_000
print(
    f"Matched {len(endpoint_df)} deterministic best-checkpoint evaluations; "
    f"episodes per run: {sorted(endpoint_df['best_eval_episodes'].unique())}"
)

with pd.option_context("display.max_colwidth", None):
    display(
        endpoint_df.sort_values(
            "best_eval_survival_pct", ascending=False
        )[
            [
                "direction_label",
                "seed",
                "compute_backend",
                "best_eval_source",
                "best_eval_step_m",
                "best_eval_episodes",
                "best_eval_survival_pct",
                "best_eval_json",
            ]
        ].round(2)
    )
print("done")

Found 27 old full-test JSON file(s) under /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/full_test_eval
Found 13 continuation full-test JSON file(s) under /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/full_test_eval/continuation_20260731
Selected full-test source counts:


,best_eval_source,runs
0,continuation,13
1,old,14


,direction_label,seed,run_name,best_eval_step_old,best_eval_survival_pct_old,best_eval_step_continuation,best_eval_survival_pct_continuation,continuation_delta_pp,continuation_step_delta_m
0,g=a2b · l=a2b,0,gs_hmd_hetero_n0_none_ga2b_la2b_s0,4478976.0,69.73,14183424.0,69.60,-0.13,9.70
3,g=a2b · l=b2a,0,gs_hmd_hetero_n0_none_ga2b_lb2a_s0,12109824.0,98.09,14515200.0,98.61,0.52,2.41
4,g=a2b · l=b2a,1,gs_hmd_hetero_n0_none_ga2b_lb2a_s1,13934592.0,96.66,14681088.0,99.03,2.37,0.75
6,g=a2b · l=bi,0,gs_hmd_hetero_n0_none_ga2b_lbi_s0,12607488.0,95.28,14929920.0,97.19,1.91,2.32
10,g=b2a · l=a2b,1,gs_hmd_hetero_n0_none_gb2a_la2b_s1,13602816.0,96.92,14846976.0,96.96,0.04,1.24
12,g=b2a · l=b2a,0,gs_hmd_hetero_n0_none_gb2a_lb2a_s0,9289728.0,90.05,14017536.0,95.40,5.35,4.73
13,g=b2a · l=b2a,1,gs_hmd_hetero_n0_none_gb2a_lb2a_s1,12026880.0,99.66,14929920.0,99.25,-0.41,2.90
14,g=b2a · l=b2a,2,gs_hmd_hetero_n0_none_gb2a_lb2a_s2,12358656.0,99.63,14017536.0,99.18,-0.45,1.66
16,g=b2a · l=bi,1,gs_hmd_hetero_n0_none_gb2a_lbi_s1,10616832.0,97.24,14598144.0,90.21,-7.03,3.98
18,g=bi · l=a2b,0,gs_hmd_hetero_n0_none_gbi_la2b_s0,11695104.0,87.73,14929920.0,86.52,-1.21,3.23


Matched 27 deterministic best-checkpoint evaluations; episodes per run: [201]


,direction_label,seed,compute_backend,best_eval_source,best_eval_step_m,best_eval_episodes,best_eval_survival_pct,best_eval_json
23,g=bi · l=b2a,2,JED (CPU),old,14.93,201,99.53,outputs/full_test_eval/best_test_gs_hmd_hetero_n0_none_gbi_lb2a_s2_step14929920_job3096906.json
13,g=b2a · l=b2a,1,IZAR (GPU),continuation,14.93,201,99.25,outputs/full_test_eval/continuation_20260731/best_test_gs_hmd_hetero_n0_none_gb2a_lb2a_s1.json
14,g=b2a · l=b2a,2,JED (CPU),continuation,14.02,201,99.18,outputs/full_test_eval/continuation_20260731/best_test_gs_hmd_hetero_n0_none_gb2a_lb2a_s2.json
11,g=b2a · l=a2b,2,IZAR (GPU),old,14.85,201,99.03,outputs/full_test_eval/best_test_gs_hmd_hetero_n0_none_gb2a_la2b_s2_step14846976_job3096894.json
4,g=a2b · l=b2a,1,JED (CPU),continuation,14.68,201,99.03,outputs/full_test_eval/continuation_20260731/best_test_gs_hmd_hetero_n0_none_ga2b_lb2a_s1.json
3,g=a2b · l=b2a,0,IZAR (GPU),continuation,14.52,201,98.61,outputs/full_test_eval/continuation_20260731/best_test_gs_hmd_hetero_n0_none_ga2b_lb2a_s0.json
24,g=bi · l=bi,0,JED (CPU),continuation,14.76,201,98.40,outputs/full_test_eval/continuation_20260731/best_test_gs_hmd_hetero_n0_none_gbi_lbi_s0.json
19,g=bi · l=a2b,1,JED (CPU),old,14.43,201,97.40,outputs/full_test_eval/best_test_gs_hmd_hetero_n0_none_gbi_la2b_s1_step14432256_job3096902.json
6,g=a2b · l=bi,0,JED (CPU),continuation,14.93,201,97.19,outputs/full_test_eval/continuation_20260731/best_test_gs_hmd_hetero_n0_none_ga2b_lbi_s0.json
10,g=b2a · l=a2b,1,JED (CPU),continuation,14.85,201,96.96,outputs/full_test_eval/continuation_20260731/best_test_gs_hmd_hetero_n0_none_gb2a_la2b_s1.json


done


## Aggregated survival curves

One curve per direction pair, aggregated over its three seeds. Thin lines are
the individual seeds; the band is one standard deviation across them. The
baseline is drawn in grey.

In [8]:
DIRECTION_COLORS = {
    BASELINE_LABEL: "#6b7280",
    "g=bi · l=a2b": "#1f77b4",
    "g=bi · l=b2a": "#17becf",
    "g=a2b · l=bi": "#2ca02c",
    "g=a2b · l=a2b": "#d62728",
    "g=a2b · l=b2a": "#ff7f0e",
    "g=b2a · l=bi": "#9467bd",
    "g=b2a · l=a2b": "#8c564b",
    "g=b2a · l=b2a": "#e377c2",
}

absolute_figure = sc.plot_survival_comparison(
    survival_long,
    group_by=["generator_code", "load_code"],
    label_by=direction_curve_label,
    smooth=SMOOTH_WINDOW,
    uncertainty="std",
    min_members=1,
    show_members=True,
    colors=DIRECTION_COLORS,
    highlight=[BASELINE_LABEL],
    budget_step=comparison_budget,
    title=(
        "gs_hmd: test episodic survival by generator / load message direction"
    ),
    width=1450,
    height=700,
)
absolute_figure.show()
print("done")

done


### Small multiples by generator direction

The same data faceted so each panel holds one generator direction and compares
the three load directions inside it.

In [9]:
faceted_figure = sc.plot_survival_comparison(
    survival_long,
    group_by=["generator_code", "load_code"],
    label_by=lambda row: f'l={row["load_code"]}',
    facet_by="generator_direction",
    facet_order=DIRECTION_ORDER,
    smooth=SMOOTH_WINDOW,
    uncertainty="std",
    min_members=1,
    budget_step=comparison_budget,
    title="gs_hmd: load direction within each generator direction",
    width=1450,
    height=520,
    ncols=3,
)
faceted_figure.show()
print("done")

done


### Small multiples by load direction

The transpose of the previous view: each panel fixes one load direction and
compares the three generator directions inside it. Read together, the two
faceted views separate a main effect from an interaction -- if the same
generator direction wins in all three load panels, the generator relation is
acting independently of the load relation.

In [10]:
generator_within_load_figure = sc.plot_survival_comparison(
    survival_long,
    group_by=["generator_code", "load_code"],
    label_by=lambda row: f'g={row["generator_code"]}',
    facet_by="load_direction",
    facet_order=DIRECTION_ORDER,
    smooth=SMOOTH_WINDOW,
    uncertainty="std",
    min_members=1,
    budget_step=comparison_budget,
    title="gs_hmd: generator direction within each load direction",
    width=1450,
    height=520,
    ncols=3,
)
generator_within_load_figure.show()
print("done")

done


### One panel per relation, each against the baseline

The same editable `wm.plot_run_mean_groups` layout used in the GINE search
notebook: one subplot per direction pair, each showing that pair against the
`gbi_lbi` baseline. Thin lines are the individual seeds, the thick line is the
seed mean, and the band is one standard deviation.

`HMD_SUBPLOTS` is a plain dict, so panels can be removed, reordered, or added
by hand after it is built. Each entry accepts the usual selectors --
`prefix`, `name`, `runs`, `contains`, `regex`, `run_dir` -- plus `label`,
`color`, and `width`.

In [11]:
HMD_RUN_PREFIX = "gs_hmd_hetero_n0_none"

baseline_generator_code = DIRECTION_CODES[BASELINE_GENERATOR_DIRECTION]
baseline_load_code = DIRECTION_CODES[BASELINE_LOAD_DIRECTION]
direction_code_order = [DIRECTION_CODES[name] for name in DIRECTION_ORDER]


def hmd_run_prefix(generator_code, load_code):
    """Prefix shared by the three seeds of one direction pair."""
    return f"{HMD_RUN_PREFIX}_g{generator_code}_l{load_code}_s"


HMD_BASELINE_SPEC = {
    "prefix": hmd_run_prefix(baseline_generator_code, baseline_load_code),
    "label": BASELINE_LABEL,
    "color": "#6b7280",
    "width": 4,
}

# Built from the factorial so a renamed or reseeded config cannot silently
# drop a panel. Edit the dict afterwards to drop or reorder panels.
HMD_SUBPLOTS = {}
for generator_code_value in direction_code_order:
    for load_code_value in direction_code_order:
        if (generator_code_value, load_code_value) == (
            baseline_generator_code,
            baseline_load_code,
        ):
            continue
        variant_label = f"g={generator_code_value} · l={load_code_value}"
        HMD_SUBPLOTS[f"{BASELINE_LABEL}  vs  {variant_label}"] = [
            HMD_BASELINE_SPEC,
            {
                "prefix": hmd_run_prefix(
                    generator_code_value, load_code_value
                ),
                "label": variant_label,
                "color": "#d62728",
            },
        ]

print(f"{len(HMD_SUBPLOTS)} panels requested")

HMD_MEAN_GROUPS = {
    title: wm.resolve_named_plot_specs(run_specs, history=history_df)
    for title, run_specs in HMD_SUBPLOTS.items()
}
# A panel with fewer than two resolved specs would show the baseline alone,
# which reads as a real comparison but is not one.
dropped = [title for title, specs in HMD_MEAN_GROUPS.items() if len(specs) < 2]
HMD_MEAN_GROUPS = {
    title: specs for title, specs in HMD_MEAN_GROUPS.items() if len(specs) >= 2
}
if dropped:
    print(f"Dropped {len(dropped)} panel(s) with missing runs:")
    for title in dropped:
        print("  ", title)
print(f"{len(HMD_MEAN_GROUPS)} panels plotted")

hmd_relation_fig = wm.plot_run_mean_groups(
    HMD_MEAN_GROUPS,
    split="test",
    smooth=SMOOTH_WINDOW,
    title=(
        "gs_hmd: each generator / load message direction against the "
        "bidirectional baseline"
    ),
    title_font_size=26,
    subplot_title_font_size=20,
    ncols=4,
    subplot_height=470,
    width=2400,
    y_range=[0, 105],
    show_members=True,
    show_std=True,
    save_name="gs_hmd_direction_vs_baseline_subplots",
    history=history_df,
    horizontal_spacing=0.045,
    vertical_spacing=0.09,
    margin={"l": 50, "r": 20, "t": 90, "b": 45},
)
# Explicit show(): the cell ends with print("done"), so a bare
# expression here would not be auto-displayed.
hmd_relation_fig.show()
print("done")

8 panels requested
8 panels plotted
Plot source folders used to build curves: 1 folder(s), 27 run(s)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/gs_hmd (27 runs)
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/gs_hmd_direction_vs_baseline_subplots.html


done


## Direction heatmaps

Cell statistic: each run contributes the deterministic full-test survival of
its saved best checkpoint. The three seeds sharing a direction cell are then
averaged so every run has equal weight. This does not select the maximum test
value post hoc and does not average the final seven training evaluations.

Three views are produced:

1. **absolute** mean survival per direction pair;
2. **difference from the baseline** cell, on a diverging scale;
3. **seed count** per cell, to confirm each cell really has three runs.

Each heatmap is followed by the exact configs behind it.

In [12]:
HEATMAP_VALUE_COLUMN = "best_eval_survival_pct"
heatmap_endpoint_df = endpoint_df.dropna(
    subset=[HEATMAP_VALUE_COLUMN]
).copy()
excluded = len(endpoint_df) - len(heatmap_endpoint_df)
if excluded:
    print(f"Excluded {excluded} run(s) without a valid full-test result.")

DIRECTION_CODE_ORDER = [DIRECTION_CODES[name] for name in DIRECTION_ORDER]


def direction_table(frame, value_column, aggfunc="mean"):
    table = frame.pivot_table(
        index="generator_code",
        columns="load_code",
        values=value_column,
        aggfunc=aggfunc,
    )
    return table.reindex(
        index=DIRECTION_CODE_ORDER, columns=DIRECTION_CODE_ORDER
    ).rename_axis(index="generator", columns="load")


def show_direction_heatmap(
    table,
    title,
    color_label,
    colorscale="Viridis",
    zmin=None,
    zmax=None,
    zmid=None,
    text_format=".1f",
    source_df=None,
):
    if table.empty or table.notna().sum().sum() == 0:
        print(f"No data available for: {title}")
        return None
    print(title)
    display(table.round(2))
    kwargs = dict(
        text_auto=text_format,
        aspect="auto",
        color_continuous_scale=colorscale,
        labels={
            "x": "Load direction",
            "y": "Generator direction",
            "color": color_label,
        },
        title=title,
    )
    if zmin is not None:
        kwargs["zmin"] = zmin
    if zmax is not None:
        kwargs["zmax"] = zmax
    if zmid is not None:
        kwargs["color_continuous_midpoint"] = zmid
    figure = px.imshow(table, **kwargs)
    figure.show()
    if source_df is not None:
        detail = (
            source_df[
                [
                    "generator_code",
                    "load_code",
                    "seed",
                    "compute_backend",
                    "run_name",
                    "config_path",
                    "best_eval_step_m",
                    "best_eval_episodes",
                    "best_eval_source",
                    "best_eval_json",
                    HEATMAP_VALUE_COLUMN,
                ]
            ]
            .sort_values(["generator_code", "load_code", "seed"])
            .reset_index(drop=True)
        )
        print(f"Configs used for: {title}")
        with pd.option_context("display.max_colwidth", None):
            display(detail.round(2))
    return figure


absolute_table = direction_table(heatmap_endpoint_df, HEATMAP_VALUE_COLUMN)
show_direction_heatmap(
    absolute_table,
    "Generator × load message direction — full-test survival of best checkpoint",
    "Mean survival (%)",
    colorscale="Viridis",
    zmin=0,
    zmax=100,
    source_df=heatmap_endpoint_df,
)
print("done")

Generator × load message direction — full-test survival of best checkpoint


load,bi,a2b,b2a
generator,,,
bi,92.65,91.35,87.76
a2b,83.16,56.41,94.47
b2a,84.93,97.43,97.94


Configs used for: Generator × load message direction — full-test survival of best checkpoint


,generator_code,load_code,seed,compute_backend,run_name,config_path,best_eval_step_m,best_eval_episodes,best_eval_source,best_eval_json,best_eval_survival_pct
0,a2b,a2b,0,IZAR (GPU),gs_hmd_hetero_n0_none_ga2b_la2b_s0,configs/gnn_graph_screening/heterogenous_message_direction/gs_hmd_hetero_n0_none_ga2b_la2b_s0.toml,14.18,201,continuation,outputs/full_test_eval/continuation_20260731/best_test_gs_hmd_hetero_n0_none_ga2b_la2b_s0.json,69.60
1,a2b,a2b,1,JED (CPU),gs_hmd_hetero_n0_none_ga2b_la2b_s1,configs/gnn_graph_screening/heterogenous_message_direction/gs_hmd_hetero_n0_none_ga2b_la2b_s1.toml,4.31,201,old,outputs/full_test_eval/best_test_gs_hmd_hetero_n0_none_ga2b_la2b_s1_step4313088_job3096884.json,68.76
2,a2b,a2b,2,IZAR (GPU),gs_hmd_hetero_n0_none_ga2b_la2b_s2,configs/gnn_graph_screening/heterogenous_message_direction/gs_hmd_hetero_n0_none_ga2b_la2b_s2.toml,12.86,201,old,outputs/full_test_eval/best_test_gs_hmd_hetero_n0_none_ga2b_la2b_s2_step12856320_job3096885.json,30.88
3,a2b,b2a,0,IZAR (GPU),gs_hmd_hetero_n0_none_ga2b_lb2a_s0,configs/gnn_graph_screening/heterogenous_message_direction/gs_hmd_hetero_n0_none_ga2b_lb2a_s0.toml,14.52,201,continuation,outputs/full_test_eval/continuation_20260731/best_test_gs_hmd_hetero_n0_none_ga2b_lb2a_s0.json,98.61
4,a2b,b2a,1,JED (CPU),gs_hmd_hetero_n0_none_ga2b_lb2a_s1,configs/gnn_graph_screening/heterogenous_message_direction/gs_hmd_hetero_n0_none_ga2b_lb2a_s1.toml,14.68,201,continuation,outputs/full_test_eval/continuation_20260731/best_test_gs_hmd_hetero_n0_none_ga2b_lb2a_s1.json,99.03
5,a2b,b2a,2,JED (CPU),gs_hmd_hetero_n0_none_ga2b_lb2a_s2,configs/gnn_graph_screening/heterogenous_message_direction/gs_hmd_hetero_n0_none_ga2b_lb2a_s2.toml,9.87,201,old,outputs/full_test_eval/best_test_gs_hmd_hetero_n0_none_ga2b_lb2a_s2_step9870336_job3096888.json,85.78
6,a2b,bi,0,JED (CPU),gs_hmd_hetero_n0_none_ga2b_lbi_s0,configs/gnn_graph_screening/heterogenous_message_direction/gs_hmd_hetero_n0_none_ga2b_lbi_s0.toml,14.93,201,continuation,outputs/full_test_eval/continuation_20260731/best_test_gs_hmd_hetero_n0_none_ga2b_lbi_s0.json,97.19
7,a2b,bi,1,JED (CPU),gs_hmd_hetero_n0_none_ga2b_lbi_s1,configs/gnn_graph_screening/heterogenous_message_direction/gs_hmd_hetero_n0_none_ga2b_lbi_s1.toml,14.85,201,old,outputs/full_test_eval/best_test_gs_hmd_hetero_n0_none_ga2b_lbi_s1_step14846976_job3096890.json,72.27
8,a2b,bi,2,IZAR (GPU),gs_hmd_hetero_n0_none_ga2b_lbi_s2,configs/gnn_graph_screening/heterogenous_message_direction/gs_hmd_hetero_n0_none_ga2b_lbi_s2.toml,14.68,201,old,outputs/full_test_eval/best_test_gs_hmd_hetero_n0_none_ga2b_lbi_s2_step14681088_job3096891.json,80.01
9,b2a,a2b,0,JED (CPU),gs_hmd_hetero_n0_none_gb2a_la2b_s0,configs/gnn_graph_screening/heterogenous_message_direction/gs_hmd_hetero_n0_none_gb2a_la2b_s0.toml,14.93,201,old,outputs/full_test_eval/best_test_gs_hmd_hetero_n0_none_gb2a_la2b_s0_step14929920_job3096892.json,96.32


done


In [13]:
baseline_code = (
    DIRECTION_CODES[BASELINE_GENERATOR_DIRECTION],
    DIRECTION_CODES[BASELINE_LOAD_DIRECTION],
)
baseline_value = absolute_table.loc[baseline_code[0], baseline_code[1]]
if pd.isna(baseline_value):
    raise ValueError(
        "The baseline cell has no value, so a relative heatmap is undefined."
    )
print(f"Baseline ({BASELINE_LABEL}) = {baseline_value:.2f}%")

delta_table = absolute_table - baseline_value
delta_limit = float(np.nanmax(np.abs(delta_table.to_numpy())))
show_direction_heatmap(
    delta_table,
    f"Difference from the {BASELINE_LABEL} baseline",
    "Survival difference (pp)",
    colorscale="RdBu",
    zmin=-delta_limit,
    zmax=delta_limit,
    zmid=0.0,
    text_format="+.1f",
)
print("done")

Baseline (g=bi · l=bi) = 92.65%
Difference from the g=bi · l=bi baseline


load,bi,a2b,b2a
generator,,,
bi,0.00,-1.30,-4.89
a2b,-9.49,-36.24,1.83
b2a,-7.72,4.79,5.30


done


In [14]:
seed_count_table = direction_table(
    heatmap_endpoint_df, "run_name", aggfunc="nunique"
)
show_direction_heatmap(
    seed_count_table,
    "Runs contributing to each cell",
    "Runs",
    colorscale="Blues",
    zmin=0,
    text_format="d",
)

spread_table = direction_table(
    heatmap_endpoint_df, HEATMAP_VALUE_COLUMN, aggfunc="std"
)
show_direction_heatmap(
    spread_table,
    "Seed-to-seed standard deviation per cell",
    "Std across seeds (pp)",
    colorscale="Oranges",
    zmin=0,
)
print("done")

Runs contributing to each cell


load,bi,a2b,b2a
generator,,,
bi,3,3,3
a2b,3,3,3
b2a,3,3,3


Seed-to-seed standard deviation per cell


load,bi,a2b,b2a
generator,,,
bi,8.24,5.54,11.04
a2b,12.75,22.12,7.53
b2a,13.01,1.42,2.20


done


### Heatmap with the individual seed values

Same cells and same colour scale as the direction heatmap above, but each cell
also prints its three per-seed values underneath the mean. This is the quickest
way to tell a real difference from one lucky seed: a cell whose mean is high
only because one seed is far above the other two is not a finding.

Seeds are labelled explicitly (`s0`, `s1`, `s2`) rather than positionally, so a
missing seed is visible instead of silently shifting the order. The statistic is
whatever `HEATMAP_VALUE_COLUMN` is set to above, so these stay consistent with
the other heatmaps.

In [15]:
def heatmap_text_color(value, vmin, vmax):
    """White on the dark end of the scale, near-black on the light end."""
    if pd.isna(value):
        return "#111827"
    span = (vmax - vmin) or 1.0
    return "#f9fafb" if (value - vmin) / span < 0.55 else "#111827"


def show_heatmap_with_seeds(
    frame,
    value_column,
    index_column,
    column_column,
    index_order,
    column_order,
    title,
    x_label,
    y_label,
    color_label,
    colorscale="Viridis",
    zmin=0,
    zmax=100,
    seed_column="seed",
    mean_font_size=22,
    seed_font_size=11,
    width=980,
):
    mean_table = frame.pivot_table(
        index=index_column,
        columns=column_column,
        values=value_column,
        aggfunc="mean",
    ).reindex(index=index_order, columns=column_order)
    if mean_table.notna().sum().sum() == 0:
        print(f"No data available for: {title}")
        return None

    seed_labels = {}
    for (row_key, column_key), group in frame.groupby(
        [index_column, column_column]
    ):
        ordered = group.sort_values(seed_column)
        seed_labels[(row_key, column_key)] = "   ".join(
            f"s{int(seed)} {value:.1f}"
            for seed, value in zip(
                ordered[seed_column], ordered[value_column]
            )
        )

    figure = go.Figure(
        go.Heatmap(
            z=mean_table.to_numpy(),
            x=[str(value) for value in mean_table.columns],
            y=[str(value) for value in mean_table.index],
            colorscale=colorscale,
            zmin=zmin,
            zmax=zmax,
            colorbar={"title": color_label},
            hovertemplate=(
                f"{y_label}: %{{y}}<br>{x_label}: %{{x}}<br>"
                f"mean {color_label}: %{{z:.2f}}<extra></extra>"
            ),
        )
    )
    for row_index, row_key in enumerate(mean_table.index):
        for column_index, column_key in enumerate(mean_table.columns):
            value = mean_table.iloc[row_index, column_index]
            if pd.isna(value):
                continue
            color = heatmap_text_color(value, zmin, zmax)
            figure.add_annotation(
                x=str(column_key),
                y=str(row_key),
                text=f"<b>{value:.1f}</b>",
                showarrow=False,
                yshift=15,
                font={"size": mean_font_size, "color": color},
            )
            label = seed_labels.get((row_key, column_key))
            if label:
                figure.add_annotation(
                    x=str(column_key),
                    y=str(row_key),
                    text=label,
                    showarrow=False,
                    yshift=-16,
                    font={"size": seed_font_size, "color": color},
                )

    figure.update_layout(
        title=title,
        xaxis_title=x_label,
        yaxis_title=y_label,
        width=width,
        height=150 * len(mean_table.index) + 200,
        # Match px.imshow, which puts the first row at the top.
        yaxis={"autorange": "reversed"},
    )
    figure.show()
    return mean_table


seeded_absolute_table = show_heatmap_with_seeds(
    heatmap_endpoint_df,
    HEATMAP_VALUE_COLUMN,
    "generator_code",
    "load_code",
    DIRECTION_CODE_ORDER,
    DIRECTION_CODE_ORDER,
    "Generator × load direction — cell mean with the three seed values",
    "Load direction",
    "Generator direction",
    "Mean survival (%)",
    colorscale="Viridis",
    zmin=0,
    zmax=100,
)
display(seeded_absolute_table.round(2))
print("done")

load_code,bi,a2b,b2a
generator_code,,,
bi,92.65,91.35,87.76
a2b,83.16,56.41,94.47
b2a,84.93,97.43,97.94


done


The same view with all nine direction pairs laid out along one axis, which
avoids having to mentally recombine the generator and load codes.

## Direction-pair ranking

Seed-aggregated ranking from the deterministic full-test evaluation of each
run's saved best checkpoint, with the baseline delta attached. A pair only
beats the baseline convincingly if `mean_minus_baseline` is positive and
comfortably larger than `std_survival_pct`.

In [16]:
valid_endpoint_df = endpoint_df.dropna(
    subset=["best_eval_survival_pct"]
).copy()

ranking = (
    valid_endpoint_df.groupby(
        ["generator_code", "load_code", "direction_label"],
        as_index=False,
    )
    .agg(
        mean_survival_pct=("best_eval_survival_pct", "mean"),
        std_survival_pct=("best_eval_survival_pct", "std"),
        min_survival_pct=("best_eval_survival_pct", "min"),
        max_survival_pct=("best_eval_survival_pct", "max"),
        seeds=("seed", "nunique"),
        backends=("compute_backend", lambda x: ", ".join(sorted(set(x)))),
        eval_sources=("best_eval_source", lambda x: ", ".join(sorted(set(x)))),
    )
    .sort_values("mean_survival_pct", ascending=False)
)

baseline_mask = (
    ranking["generator_code"].eq(baseline_code[0])
    & ranking["load_code"].eq(baseline_code[1])
)
if not baseline_mask.any():
    raise ValueError("The baseline pair is missing from the ranking.")
baseline_mean = float(ranking.loc[baseline_mask, "mean_survival_pct"].iloc[0])
ranking["mean_minus_baseline"] = ranking["mean_survival_pct"] - baseline_mean

display(ranking.round(2))

rank_plot = px.bar(
    ranking.sort_values("mean_minus_baseline"),
    x="mean_minus_baseline",
    y="direction_label",
    orientation="h",
    color="mean_minus_baseline",
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0.0,
    hover_data=["mean_survival_pct", "std_survival_pct", "seeds"],
    title=(
        "Full-test survival of each run's best checkpoint, "
        f"relative to {BASELINE_LABEL}"
    ),
    labels={
        "mean_minus_baseline": "Difference from baseline (pp)",
        "direction_label": "Direction pair",
    },
    height=520,
)
rank_plot.add_vline(x=0.0, line_dash="dot")
rank_plot.show()
print("done")

,generator_code,load_code,direction_label,mean_survival_pct,std_survival_pct,min_survival_pct,max_survival_pct,seeds,backends,eval_sources,mean_minus_baseline
4,b2a,b2a,g=b2a · l=b2a,97.94,2.20,95.40,99.25,3,"IZAR (GPU), JED (CPU)",continuation,5.30
3,b2a,a2b,g=b2a · l=a2b,97.43,1.42,96.32,99.03,3,"IZAR (GPU), JED (CPU)","continuation, old",4.79
1,a2b,b2a,g=a2b · l=b2a,94.47,7.53,85.78,99.03,3,"IZAR (GPU), JED (CPU)","continuation, old",1.83
8,bi,bi,g=bi · l=bi,92.65,8.24,83.21,98.40,3,"IZAR (GPU), JED (CPU)","continuation, old",0.00
6,bi,a2b,g=bi · l=a2b,91.35,5.54,86.52,97.40,3,"IZAR (GPU), JED (CPU)","continuation, old",-1.30
7,bi,b2a,g=bi · l=b2a,87.76,11.04,77.64,99.53,3,"IZAR (GPU), JED (CPU)","continuation, old",-4.89
5,b2a,bi,g=b2a · l=bi,84.93,13.01,70.11,94.46,3,"IZAR (GPU), JED (CPU)","continuation, old",-7.72
2,a2b,bi,g=a2b · l=bi,83.16,12.75,72.27,97.19,3,"IZAR (GPU), JED (CPU)","continuation, old",-9.49
0,a2b,a2b,g=a2b · l=a2b,56.41,22.12,30.88,69.60,3,"IZAR (GPU), JED (CPU)","continuation, old",-36.24


done


## Marginal direction effects

Averaging over the other relation. These are exploratory marginals, not causal
estimates, but the design here is a balanced full factorial with equal seeds
per cell, so they are more trustworthy than the equivalent tables in the
combined Stage 1 notebook.

In [17]:
for factor, label in [
    ("generator_direction", "Generator direction"),
    ("load_direction", "Load direction"),
]:
    summary = (
        valid_endpoint_df.groupby(factor, as_index=False)
        .agg(
            mean_survival_pct=("best_eval_survival_pct", "mean"),
            std_survival_pct=("best_eval_survival_pct", "std"),
            runs=("run_name", "nunique"),
        )
        .sort_values("mean_survival_pct", ascending=False)
    )
    print(label)
    display(summary.round(2))
print("done")

Generator direction


,generator_direction,mean_survival_pct,std_survival_pct,runs
2,busbar_to_asset,93.43,9.21,9
1,bidirectional,90.58,7.74,9
0,asset_to_busbar,78.01,21.53,9


Load direction


,load_direction,mean_survival_pct,std_survival_pct,runs
2,busbar_to_asset,93.39,8.12,9
1,bidirectional,86.91,10.91,9
0,asset_to_busbar,81.73,22.32,9


done


## Optional CSV export

In [18]:
EXPORT_TABLES = False

if EXPORT_TABLES:
    export_dir = wm.TASK_DIR / "outputs" / "gs_hmd_direction_summary"
    export_dir.mkdir(parents=True, exist_ok=True)
    coverage.to_csv(export_dir / "coverage.csv", index=False)
    endpoint_df.to_csv(export_dir / "run_endpoints.csv", index=False)
    full_test_all_results.to_csv(export_dir / "full_test_all_results.csv", index=False)
    ranking.to_csv(export_dir / "direction_ranking.csv", index=False)
    absolute_table.to_csv(export_dir / "heatmap_absolute.csv")
    delta_table.to_csv(export_dir / "heatmap_baseline_delta.csv")
    print("Saved tables under", export_dir)
else:
    print("Set EXPORT_TABLES = True to write CSVs.")
print("done")

Set EXPORT_TABLES = True to write CSVs.
done
